# Delta Lake — cenário de pedidos

## Domínio (fonte de dados)

Pedidos de uma loja online: cada registro tem **id**, **cliente** e **valor total**. Os dados são **sintéticos** (criados no notebook); a mesma ideia vale para CSV/Parquet exportado de um sistema real.

## Modelo ER (conceitual)

- **Pedido** (entidade): `id` (PK), `cliente`, `total`.

*(O enunciado pede imagens do ER: exporte o diagrama e versione em `docs/assets/` se quiser nota completa.)*

```mermaid
erDiagram
    PEDIDO {
        int id PK
        string cliente
        double total
    }
```

## DDL (lógico)

```sql
CREATE TABLE pedidos_delta (
  id INT,
  cliente STRING,
  total DOUBLE
) USING DELTA;
```

Nos blocos abaixo: sessão Spark com Delta, criação da tabela, **INSERT**, **UPDATE** e **DELETE**.


In [1]:
import os
from pathlib import Path

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

# Windows: Spark precisa de HADOOP_HOME com winutils.exe + hadoop.dll (pasta tools/hadoop deste repo).
_root = Path.cwd().resolve()
if not (_root / "tools" / "hadoop" / "bin" / "winutils.exe").exists():
    _root = _root.parent
_hadoop = _root / "tools" / "hadoop"
if _hadoop.joinpath("bin/winutils.exe").exists():
    os.environ["HADOOP_HOME"] = str(_hadoop)
    _bin = str(_hadoop / "bin")
    os.environ["PATH"] = _bin + os.pathsep + os.environ.get("PATH", "")

warehouse = Path("data/spark-warehouse").resolve()
warehouse.mkdir(parents=True, exist_ok=True)

_builder = SparkSession.builder.appName("trabalho-delta").master("local[*]")
_builder = configure_spark_with_delta_pip(_builder)
_builder = _builder.config("spark.sql.warehouse.dir", str(warehouse))
_builder = _builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
_builder = _builder.config(
    "spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"
)
spark = _builder.enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel("WARN")


In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS trabalho")
spark.sql("USE trabalho")
spark.sql("DROP TABLE IF EXISTS pedidos_delta")
spark.sql("""
CREATE TABLE pedidos_delta (
  id INT,
  cliente STRING,
  total DOUBLE
) USING DELTA
""")


DataFrame[]

In [3]:
# INSERT: incluir linhas
spark.sql("""
INSERT INTO pedidos_delta VALUES
  (1, 'Ana', 100.0),
  (2, 'Beto', 200.0),
  (3, 'Carla', 50.0)
""")
spark.sql("SELECT * FROM pedidos_delta ORDER BY id").show()


+---+-------+-----+
| id|cliente|total|
+---+-------+-----+
|  1|    Ana|100.0|
|  2|   Beto|200.0|
|  3|  Carla| 50.0|
+---+-------+-----+



In [4]:
# UPDATE: alterar valores que atendem a condicao
spark.sql("UPDATE pedidos_delta SET total = 120.0 WHERE id = 1")
spark.sql("SELECT * FROM pedidos_delta ORDER BY id").show()


+---+-------+-----+
| id|cliente|total|
+---+-------+-----+
|  1|    Ana|120.0|
|  2|   Beto|200.0|
|  3|  Carla| 50.0|
+---+-------+-----+



In [5]:
# DELETE: remover linhas
spark.sql("DELETE FROM pedidos_delta WHERE id = 2")
spark.sql("SELECT * FROM pedidos_delta ORDER BY id").show()


+---+-------+-----+
| id|cliente|total|
+---+-------+-----+
|  1|    Ana|120.0|
|  3|  Carla| 50.0|
+---+-------+-----+

